In [11]:
import os
import json
import time
import random
import urllib.request
import urllib.error
from pathlib import Path
import yaml

In [13]:
cwd = Path.cwd()
CONFIG_PATH = cwd.parent / "config" / "config.yaml"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

BASE_DIR = Path(cfg["paths"]["htem_raw_data_root"]).resolve()
assert BASE_DIR.exists(), f"Filtered path does not exist: {BASE_DIR}"

LIBRARY_URL_TEMPLATE = "https://htem-api.nrel.gov/api/sample_library/{}"
SAMPLE_URL_TEMPLATE  = "https://htem-api.nrel.gov/api/sample/{}"

SLEEP_BETWEEN_REQUESTS_SEC = 0.10
JITTER_SEC = 0.10
TIMEOUT_SEC = 30

MAX_RETRIES = 3
BACKOFF_BASE_SEC = 1.0

HEADERS = {
    "User-Agent": "HTEM-downloader/1.0 (personal research; contact: dan)"
}

In [3]:
full_final_lib_ids = [9966,
 12585,
 12589,
 6863,
 6870,
 6912,
 7069,
 7079,
 7083,
 9858,
 7541,
 10904,
 9940,
 7242,
 10936,
 7461,
 7536,
 12606,
 6706,
 6677,
 6728,
 6851,
 6852,
 6914,
 6999,
 7060,
 7117,
 7139,
 7174,
 7333,
 7548,
 7520,
 7524,
 7528,
 9969,
 9973,
 6678,
 6679,
 6691,
 6693,
 6738,
 6782,
 6790,
 6811,
 6880,
 6904,
 6934,
 6963,
 6985,
 7045,
 7095,
 7116,
 7118,
 7153,
 7172,
 7181,
 7182,
 7338,
 7345,
 7361,
 7412,
 7446,
 7449,
 7494,
 7899,
 8335,
 8336,
 8337,
 8338,
 8551,
 10104,
 10107,
 10137,
 6658,
 6684,
 7532,
 7542,
 7150,
 10136,
 7564,
 9764,
 7204,
 7529,
 6940,
 6651,
 7540,
 6923,
 6931,
 9777,
 6876,
 6701,
 7521,
 7123,
 10105,
 7551,
 7533,
 7188,
 6710,
 7113,
 10106,
 7516,
 7360,
 7523,
 7561,
 7243,
 6995,
 7505,
 8390,
 8398,
 8399,
 8405,
 8438,
 8449,
 8452,
 8454,
 8455,
 8457,
 8458,
 8464,
 8466,
 8467,
 8469,
 8526,
 10248,
 6796,
 7075,
 7082,
 7310,
 7325,
 7463,
 7486,
 8366,
 8367,
 8368,
 8369,
 8374,
 8375,
 8376,
 8377,
 8378,
 8381,
 8382,
 8383,
 8384,
 8385,
 8386,
 8387,
 8391,
 8392,
 8393,
 8394,
 8395,
 8396,
 8397,
 8404,
 8406,
 8412,
 8413,
 8414,
 8415,
 6626,
 8431,
 8432,
 8433,
 8434,
 8435,
 8436,
 8450,
 8453,
 8456,
 8462,
 8468,
 9824,
 9826,
 9859,
 9912,
 8401,
 8402,
 6672,
 8400,
 8427,
 8460,
 7402,
 10283,
 10256,
 7281,
 8451,
 8448,
 10230,
 8426,
 6915,
 8437,
 6936,
 6671,
 7439,
 7928,
 7743,
 6856,
 7073,
 8622,
 8626,
 6975,
 6648,
 7237,
 6853,
 7394,
 6630,
 6971,
 7298,
 7445,
 7002,
 7287,
 7464,
 6768,
 6944,
 7342,
 9814,
 7771,
 7417,
 7031,
 7857,
 7708,
 6699,
 7192]

In [4]:
lib_ids = full_final_lib_ids
len(lib_ids)

224

In [5]:
def fetch_json(url: str, timeout: int = TIMEOUT_SEC) -> dict:
    """
    Fetch JSON from URL with retries and exponential backoff.
    """
    last_ex = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                raw = resp.read().decode("utf-8")
            return json.loads(raw)

        except (urllib.error.HTTPError,
                urllib.error.URLError,
                TimeoutError,
                json.JSONDecodeError) as ex:
            last_ex = ex
            if attempt < MAX_RETRIES:
                sleep_s = BACKOFF_BASE_SEC * (2 ** (attempt - 1)) + random.uniform(0, 0.25)
                time.sleep(sleep_s)
            else:
                raise

    raise last_ex

In [6]:
def atomic_write_json(path: Path, obj: dict) -> None:
    """
    Write JSON safely using a temp file + replace.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")

    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

    os.replace(tmp_path, path)


def maybe_sleep():
    time.sleep(SLEEP_BETWEEN_REQUESTS_SEC + random.uniform(0, JITTER_SEC))

In [9]:
def download_libraries_and_samples(lib_ids):
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    total_libs = len(lib_ids)
    currentLibrary = 0

    cutoffLibrary = 7564
    canIterate = False

    for lib_id in lib_ids:
        currentLibrary += 1

        if canIterate or lib_id == cutoffLibrary:
            canIterate = True
        else:
            continue

        try:
            # ---- Library request ----
            lib_url = LIBRARY_URL_TEMPLATE.format(lib_id)
            library = fetch_json(lib_url)
            maybe_sleep()

            if "sample_ids" not in library or not library["sample_ids"]:
                pct = (currentLibrary / total_libs) * 100
                print(f"Library: {lib_id} SKIPPED ({pct:.2f}%)")
                continue

            # ---- Folder structure ----
            lib_folder = BASE_DIR / str(lib_id)
            samples_folder = lib_folder / "samples"
            samples_folder.mkdir(parents=True, exist_ok=True)

            # ---- Save library JSON ----
            lib_path = lib_folder / f"library {lib_id}.json"
            atomic_write_json(lib_path, library)

            numValidSamples = 0

            # ---- Sample loop ----
            for sample_id in library["sample_ids"]:
                try:
                    sample_url = SAMPLE_URL_TEMPLATE.format(sample_id)
                    sample = fetch_json(sample_url)
                    maybe_sleep()

                    sample_path = samples_folder / f"sample {sample_id}.json"
                    atomic_write_json(sample_path, sample)
                    numValidSamples += 1

                except Exception as ex:
                    print(f"Library: {lib_id} - Sample #{sample_id} Error")

            pct = (currentLibrary / total_libs) * 100
            print(
                f"Library: {lib_id} DONE ({pct:.2f}%) | "
                f"samples saved: {numValidSamples}/{len(library['sample_ids'])}"
            )

        except Exception as ex:
            pct = (currentLibrary / total_libs) * 100
            print(f"Library: {lib_id} ACCESS ERROR ({pct:.2f}%)")

In [10]:
download_libraries_and_samples(lib_ids)

Library: 7564 DONE (35.71%) | samples saved: 44/44
Library: 9764 DONE (36.16%) | samples saved: 44/44
Library: 7204 DONE (36.61%) | samples saved: 44/44
Library: 7529 DONE (37.05%) | samples saved: 44/44
Library: 6940 DONE (37.50%) | samples saved: 44/44
Library: 6651 DONE (37.95%) | samples saved: 44/44
Library: 7540 DONE (38.39%) | samples saved: 44/44
Library: 6923 DONE (38.84%) | samples saved: 44/44
Library: 6931 DONE (39.29%) | samples saved: 44/44
Library: 9777 DONE (39.73%) | samples saved: 44/44
Library: 6876 DONE (40.18%) | samples saved: 44/44
Library: 6701 DONE (40.62%) | samples saved: 44/44
Library: 7521 DONE (41.07%) | samples saved: 44/44
Library: 7123 DONE (41.52%) | samples saved: 44/44
Library: 10105 DONE (41.96%) | samples saved: 44/44
Library: 7551 DONE (42.41%) | samples saved: 44/44
Library: 7533 DONE (42.86%) | samples saved: 44/44
Library: 7188 DONE (43.30%) | samples saved: 44/44
Library: 6710 DONE (43.75%) | samples saved: 44/44
Library: 7113 DONE (44.20%) | 